# Brouillon Julien — partie 2A (données) et 2B (modèles)

Le **test n'est pas ouvert** ici : tout se compare sur la validation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

users = pd.read_csv('rev-users.csv', parse_dates=['created_date'])
notifs = pd.read_csv('rev-notifications.csv', parse_dates=['created_date'])
tx = pd.read_csv('rev-transactions.csv', parse_dates=['created_date'])

# Règles fixées en partie 1
WINDOW = pd.Timedelta(days=60)
MIN_TENURE = pd.Timedelta(days=60)
CUTOFFS = {'fit': pd.Timestamp('2018-11-14'),
           'validation': pd.Timestamp('2019-01-14'),
           'test': pd.Timestamp('2019-03-15')}
X_OFFRE, BUDGET = 5, 500          # € par offre, offres par mois (hypothèses)
INTERCHANGE, USD_TO_EUR = 0.003, 0.89

## 1. Audit des données

| Constat | Décision |
|---|---|
| 202 transactions > 100 000 $, jusqu'à 85 milliards $ (virements) | erreurs de saisie : montants **plafonnés** au 99e percentile, calculé sur la période avant la 1re coupure |
| 58 604 transactions à 0 $ | paiements annulés, en attente ou refusés (REVERTED, PENDING, DECLINED) : gardés comme **signal d'échec**, sans effet sur les montants |
| Infos marchand vides à 42 % | vides seulement hors paiements carte (complètes à 100 % pour les paiements carte) : utilisées sur les paiements carte |
| Opt-in marketing inconnus pour 34 % des clients | inconnu ≠ non : on ajoute une colonne « inconnu » |
| `num_referrals`, `num_successful_referrals` toujours à 0 | retirées |
| 664 clients sans aucune transaction | jamais actifs, donc jamais dans le tableau |
| 715 notifications datées avant l'inscription | une par client, moins de 19 h avant, notifications d'onboarding : décalage d'horloge, gardées |
| `plan` | **fuite** (valeur au jour de l'export) : retirée |

In [2]:
# Vérifications de l'audit
print('Transactions > 100 000 $ :', (tx.amount_usd > 1e5).sum())
print('Transactions à 0 $ :', (tx.amount_usd == 0).sum())
print(tx[tx.amount_usd == 0].transactions_state.value_counts().to_dict())
print('Infos marchand vides (paiements carte) :',
      f"{tx[tx.transactions_type == 'CARD_PAYMENT'].ea_merchant_mcc.isna().mean():.1%}")
print('Opt-in inconnus :', f"{users.attributes_notifications_marketing_push.isna().mean():.1%}")
print('Colonnes constantes :', [c for c in users.columns if users[c].nunique() <= 1])
print('Clients sans transaction :', (~users.user_id.isin(tx.user_id)).sum())

# Plafond des montants : 99e percentile, calculé uniquement sur le passé de la 1re coupure
CAP = tx.loc[(tx.created_date < CUTOFFS['fit']) & (tx.amount_usd > 0), 'amount_usd'].quantile(0.99)
tx['montant'] = tx.amount_usd.clip(upper=CAP)
print(f'Plafond des montants : {CAP:.0f} $')

Transactions > 100 000 $ : 202
Transactions à 0 $ : 58604


{'REVERTED': 31673, 'PENDING': 10747, 'DECLINED': 8180, 'COMPLETED': 8002, 'CANCELLED': 2}
Infos marchand vides (paiements carte) : 0.0%
Opt-in inconnus : 34.0%
Colonnes constantes : ['num_referrals', 'num_successful_referrals']


Clients sans transaction : 664
Plafond des montants : 851 $


## 2. `build_dataset(cutoff)` : une ligne par client actif

Toutes les features utilisent **uniquement** ce qui s'est passé avant la coupure ; le label uniquement ce qui se passe après.

In [3]:
ECHEC = ['DECLINED', 'FAILED', 'REVERTED', 'CANCELLED']
TOP_PAYS = users.country.value_counts().index[:5]

def build_dataset(cutoff):
    passe = tx[tx.created_date < cutoff]
    futur = tx[(tx.created_date >= cutoff) & (tx.created_date < cutoff + WINDOW)]
    p90 = passe[passe.created_date >= cutoff - pd.Timedelta(days=90)]
    p30 = passe[passe.created_date >= cutoff - pd.Timedelta(days=30)]
    ok90 = p90[p90.transactions_state == 'COMPLETED']
    carte90 = ok90[ok90.transactions_type == 'CARD_PAYMENT']

    # clients actifs (une transaction dans les 60 jours avant) et ancienneté >= 60 jours
    actifs = passe[passe.created_date >= cutoff - WINDOW].user_id.unique()
    u = users.set_index('user_id').loc[actifs]
    d = pd.DataFrame(index=u.index)
    d['anciennete_j'] = (cutoff - u.created_date).dt.days
    d = d[d.anciennete_j >= MIN_TENURE.days]
    idx = d.index
    par_client = lambda s: s.reindex(idx).fillna(0)

    # Activité
    d['recence_j'] = (cutoff - passe.groupby('user_id').created_date.max().reindex(idx)).dt.days
    d['nb_tx_30j'] = par_client(p30.groupby('user_id').size())
    d['nb_tx_90j'] = par_client(p90.groupby('user_id').size())
    d['tendance'] = d.nb_tx_30j / (d.nb_tx_90j / 3 + 1)

    # Montants (transactions terminées, plafonnées)
    d['montant_90j'] = par_client(ok90.groupby('user_id').montant.sum())
    d['panier_moyen_90j'] = par_client(ok90.groupby('user_id').montant.mean())

    # Types de transactions (part sur 90 jours)
    for t in ['CARD_PAYMENT', 'TRANSFER', 'TOPUP', 'EXCHANGE', 'ATM']:
        d[f'part_{t.lower()}'] = par_client(p90[p90.transactions_type == t].groupby('user_id').size()) / d.nb_tx_90j.clip(lower=1)

    # Échecs
    d['taux_echec_90j'] = par_client(p90[p90.transactions_state.isin(ECHEC)].groupby('user_id').size()) / d.nb_tx_90j.clip(lower=1)

    # Marchands (paiements carte uniquement)
    d['nb_pays_marchands_90j'] = par_client(carte90.groupby('user_id').ea_merchant_country.nunique())
    d['nb_categories_90j'] = par_client(carte90.groupby('user_id').ea_merchant_mcc.nunique())

    # Notifications reçues sur 90 jours
    n90 = notifs[(notifs.created_date < cutoff) & (notifs.created_date >= cutoff - pd.Timedelta(days=90))]
    d['nb_notifs_90j'] = par_client(n90.groupby('user_id').size())
    d['taux_echec_notifs_90j'] = par_client(n90[n90.status == 'FAILED'].groupby('user_id').size()) / d.nb_notifs_90j.clip(lower=1)

    # Profil
    d['age'] = cutoff.year - u.birth_year.reindex(idx)
    d['crypto'] = u.user_settings_crypto_unlocked.reindex(idx)
    d['nb_contacts'] = u.num_contacts.reindex(idx)
    for canal in ['push', 'email']:
        col = u[f'attributes_notifications_marketing_{canal}'].reindex(idx)
        d[f'optin_{canal}'] = col.fillna(0)
        d[f'optin_{canal}_inconnu'] = col.isna().astype(int)
    pays = u.country.reindex(idx).where(u.country.reindex(idx).isin(TOP_PAYS), 'autre')
    d = d.join(pd.get_dummies(pays, prefix='pays', dtype=int))

    # Valeur du client (€ par an) : frais + commission carte, sur 90 jours, annualisée
    frais = par_client(ok90[ok90.transactions_type == 'FEE'].groupby('user_id').montant.sum())
    carte = par_client(carte90.groupby('user_id').montant.sum())
    d['valeur_an_eur'] = (frais + INTERCHANGE * carte) * 4 * USD_TO_EUR

    # Label
    d['churn'] = (~d.index.isin(futur.user_id)).astype(int)
    return d.reset_index()

fit = build_dataset(CUTOFFS['fit'])
val = build_dataset(CUTOFFS['validation'])
print(f'fit : {len(fit)} clients, churn {fit.churn.mean():.1%}, {fit.shape[1] - 2} colonnes')
print(f'validation : {len(val)} clients, churn {val.churn.mean():.1%}')

fit : 8241 clients, churn 16.2%, 31 colonnes
validation : 10512 clients, churn 14.1%


## 3. Tableau anti-fuite : « le jour de la coupure, est-ce que je connais cette valeur ? »

| Colonne(s) | Connue à la coupure ? | Décision |
|---|---|---|
| activité, montants, types, échecs, marchands, notifications | oui : calculées uniquement sur les transactions avant la coupure | gardées |
| ancienneté, âge, pays | oui : ne changent pas (ou peu) | gardées |
| `crypto`, `nb_contacts`, opt-in | **partiellement** : photo prise à l'export (mai 2019), pas à la coupure | gardées, **limite à signaler** |
| `plan` | **non** : plan au jour de l'export, peut avoir changé à cause du départ | **retirée** |
| `num_referrals`, `num_successful_referrals` | toujours 0 | retirées |
| `valeur_an_eur` | oui : calculée sur les 90 jours avant | utilisée pour la décision, pas comme feature du modèle |

## 4. Qu'est-ce qui va avec le churn ?

Pour chaque feature, sur les clients du **fit** : taux de churn chez les 25 % les plus bas et les 25 % les plus hauts (0 contre 1 pour les colonnes oui/non).

⚠️ Ce tableau montre ce qui **va avec** le churn, pas ce qui le **cause**.

In [4]:
FEATURES = [c for c in fit.columns if c not in ['user_id', 'churn', 'valeur_an_eur']]

rows = []
for f in FEATURES:
    x = fit[f]
    if x.nunique() <= 2:
        bas, haut = fit[x == x.min()].churn.mean(), fit[x == x.max()].churn.mean()
        lb, lh = f'= {x.min():g}', f'= {x.max():g}'
    else:
        q1, q3 = x.quantile([0.25, 0.75])
        if q1 == q3:   # trop de valeurs identiques : 0 contre > 0
            bas, haut, lb, lh = fit[x <= q1].churn.mean(), fit[x > q1].churn.mean(), f'≤ {q1:g}', f'> {q1:g}'
        else:
            bas, haut, lb, lh = fit[x <= q1].churn.mean(), fit[x >= q3].churn.mean(), f'≤ {q1:.3g}', f'≥ {q3:.3g}'
    rows.append({'feature': f, 'bas': lb, 'churn si bas': f'{bas:.0%}',
                 'haut': lh, 'churn si haut': f'{haut:.0%}', 'écart (points)': round(100 * (haut - bas))})

lecture = pd.DataFrame(rows).sort_values('écart (points)', key=abs, ascending=False)
print(f'Taux de churn moyen (fit) : {fit.churn.mean():.0%}')
lecture

Taux de churn moyen (fit) : 16%


,feature,bas,churn si bas,haut,churn si haut,écart (points)
1,recence_j,≤ 1,2%,≥ 19,41%,38
2,nb_tx_30j,≤ 1,39%,≥ 25,2%,-37
3,nb_tx_90j,≤ 11,34%,≥ 73,4%,-30
4,tendance,≤ 0.283,36%,≥ 1.28,9%,-28
5,montant_90j,≤ 216,31%,≥ 2.77e+03,6%,-25
14,nb_categories_90j,≤ 2,30%,≥ 12,6%,-24
13,nb_pays_marchands_90j,≤ 1,26%,≥ 3,6%,-20
19,nb_contacts,≤ 0,27%,≥ 21,8%,-19
28,pays_RO,= 0,17%,= 1,7%,-10
26,pays_IE,= 0,17%,= 1,8%,-9


### Lecture (brouillon, à reformuler avec nos mots)

Taux de churn moyen (fit, novembre 2018) : 16 %.

1. **Le désengagement est le signal n°1.** Rien depuis 19 jours ou plus : 41 % de churn, contre 2 % pour une utilisation la veille. Même chose pour peu de transactions sur 30 jours (39 %) et pour une activité qui ralentit (36 %).
2. **Un usage étroit, c'est un usage fragile.** 2 catégories de commerce ou moins : 30 % de churn, contre 6 % au-delà de 12. Un seul pays marchand : 26 %, contre 6 % au-delà de 3. Revolut utilisé comme compte principal retient mieux qu'un usage d'appoint.
3. **Peu de liens sociaux, plus de départs.** 0 contact : 27 %, contre 8 % au-delà de 21.
4. **Le pays compte un peu.** Royaume-Uni : 21 % ; Roumanie et Irlande : 7 à 8 %.
5. **Ce qui compte peu :** taux d'échec des paiements (+4 points), âge, notifications, opt-in.

⚠️ Ce sont des associations, pas des causes. Un client qui part arrête d'utiliser sa carte, donc il a peu de catégories : c'est un **symptôme**, pas forcément une raison.

## 5. Modèles : 5 features simples contre toutes les features (validation)

In [5]:
SIMPLES = ['anciennete_j', 'recence_j', 'nb_tx_30j', 'nb_tx_90j', 'tendance']

def entraine(cols):
    logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    trees = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0)
    return {f'logistique ({len(cols)} f.)': logit.fit(fit[cols], fit.churn).predict_proba(val[cols])[:, 1],
            f'arbres ({len(cols)} f.)': trees.fit(fit[cols], fit.churn).predict_proba(val[cols])[:, 1]}

scores = {'baseline (récence)': val.recence_j}
scores.update(entraine(SIMPLES))
scores.update(entraine(FEATURES))

## 6. Décision en euros : classer par risque, ou par perte attendue ?

- **Classement par risque** : on cible les 500 clients avec le `p` le plus haut.
- **Classement par perte attendue** : on cible les 500 clients avec le `p × valeur_an_eur` le plus haut (les gros clients risqués d'abord).

Gain = Σ (churners ciblés × Z × leur valeur) − 500 × X. **Z est inconnu** : on montre 30 % et 50 %.

In [6]:
def gain(cibles, z):
    return (cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * X_OFFRE

rows = []
for nom, s in scores.items():
    v = val.assign(p=s)
    for critere, cle in [('risque', v.p), ('perte attendue', v.p * v.valeur_an_eur)]:
        if nom.startswith('baseline') and critere == 'perte attendue':
            continue
        cibles = v.loc[cle.nlargest(BUDGET).index]
        rows.append({'méthode': nom, 'classement': critere,
                     'PR-AUC': round(average_precision_score(val.churn, s), 3),
                     'churners ciblés': int(cibles.churn.sum()),
                     'gain € Z=30%': round(gain(cibles, 0.30)),
                     'gain € Z=50%': round(gain(cibles, 0.50))})
pd.DataFrame(rows)

,méthode,classement,PR-AUC,churners ciblés,gain € Z=30%,gain € Z=50%
0,baseline (récence),risque,0.385,229,-2273,-2122
1,logistique (5 f.),risque,0.405,239,-2217,-2028
2,logistique (5 f.),perte attendue,0.405,92,-1191,-318
3,arbres (5 f.),risque,0.427,275,-2097,-1828
4,arbres (5 f.),perte attendue,0.427,97,-1171,-284
5,logistique (30 f.),risque,0.450,275,-2234,-2056
6,logistique (30 f.),perte attendue,0.450,98,-1200,-333
7,arbres (30 f.),risque,0.465,291,-2078,-1796
8,arbres (30 f.),perte attendue,0.465,100,-1127,-211


### Lecture (brouillon)

1. **Plus de features, meilleur ciblage.** Avec 30 features, les arbres passent de 275 à 291 churners dans le top 500 (PR-AUC 0,427 → 0,465). Ils battent la baseline (229) et la logistique (275).
2. **Ceux qui partent sont ceux qui rapportent le moins.** Classés par risque seul, les 500 clients ciblés valent si peu que l'offre à 5 € perd environ 2 000 € par mois, même si elle retient la moitié des clients.
3. **Classer par perte attendue (`p × valeur`) aide, mais ne suffit pas avec 500 offres** : on perd encore environ 200 € à Z = 50 %.
4. **Prochaine étape :** ne plus fixer 500 offres, envoyer seulement quand `p × Z × valeur > X`.

## 7. Calibration : quand le modèle dit 40 %, est-ce que 40 % partent ?

La règle de rentabilité utilise `p` comme une vraie probabilité : il faut le vérifier (séance 4).

In [7]:
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(fit[FEATURES], fit.churn)
trees = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0).fit(fit[FEATURES], fit.churn)
probas = {'logistique': logit.predict_proba(val[FEATURES])[:, 1],
          'arbres': trees.predict_proba(val[FEATURES])[:, 1]}

tranches = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1]
calib = []
for nom, p in probas.items():
    t = pd.DataFrame({'p': p, 'churn': val.churn})
    t['tranche'] = pd.cut(t.p, tranches)
    g = t.groupby('tranche', observed=True).agg(clients=('churn', 'size'), p_moyen=('p', 'mean'), churn_reel=('churn', 'mean'))
    g.insert(0, 'modèle', nom)
    calib.append(g)
calib = pd.concat(calib).round(3)
calib

,modèle,clients,p_moyen,churn_reel
tranche,,,,
"(0.0, 0.05]",logistique,3702,0.018,0.012
"(0.05, 0.1]",logistique,1926,0.073,0.047
"(0.1, 0.2]",logistique,2048,0.143,0.147
"(0.2, 0.3]",logistique,997,0.245,0.250
"(0.3, 0.5]",logistique,1138,0.388,0.375
"(0.5, 0.7]",logistique,582,0.588,0.509
"(0.7, 1.0]",logistique,119,0.747,0.664
"(0.0, 0.05]",arbres,4420,0.018,0.011
"(0.05, 0.1]",arbres,1313,0.074,0.059


**Lecture :** les deux modèles sont bien calibrés jusqu'à 50 % (écarts de 1 à 4 points). Au-dessus, ils se surestiment un peu : la logistique annonce 59 % quand 51 % partent vraiment, les arbres 58 % pour 53 %. Les arbres sont un peu plus fiables. → On peut utiliser `p` dans la règle, en gardant en tête qu'il est légèrement optimiste pour les clients les plus risqués.

## 8. Arbres ou logistique : avantage réel ou chance ?

On répète la comparaison sur 3 périodes (entraînement sur une coupure, évaluation sur la suivante). **Le test du 15/03 n'est jamais utilisé.**

In [8]:
# 60 jours d'écart minimum : le label d'entraînement est connu avant la coupure d'évaluation
periodes = [('2018-07-16', '2018-09-14'), ('2018-09-14', '2018-11-14'), ('2018-11-14', '2019-01-14')]
datasets = {}
rows = []
for c_train, c_eval in periodes:
    for c in [c_train, c_eval]:
        if c not in datasets:
            datasets[c] = build_dataset(pd.Timestamp(c))
    tr, ev = datasets[c_train], datasets[c_eval]
    for nom, modele in [('logistique', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
                        ('arbres', HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0))]:
        p = modele.fit(tr[FEATURES], tr.churn).predict_proba(ev[FEATURES])[:, 1]
        cibles = ev.assign(p=p).nlargest(BUDGET, 'p')
        rows.append({'évaluation': c_eval, 'modèle': nom,
                     'PR-AUC': round(average_precision_score(ev.churn, p), 3),
                     'churners top 500': int(cibles.churn.sum())})
comparaison = pd.DataFrame(rows).pivot(index='évaluation', columns='modèle', values=['PR-AUC', 'churners top 500'])
comparaison

PR-AUC            churners top 500           
modèle     arbres logistique           arbres logistique
évaluation                                              
2018-09-14  0.486      0.445            277.0      269.0
2018-11-14  0.508      0.470            317.0      288.0
2019-01-14  0.465      0.450            291.0      275.0

**Lecture :** les arbres battent la logistique **sur les 3 périodes** : de +8 à +29 churners dans le top 500, et de +0,02 à +0,04 de PR-AUC. L'écart va toujours dans le même sens, donc ce n'est pas de la chance. → **Modèle retenu : arbres boostés.**

Les coupures sont espacées de 60 jours, donc aucun label d'entraînement ne chevauche la période évaluée. La performance varie d'une période à l'autre (277 à 317 churners) : le comportement des clients évolue, il faudra réentraîner le modèle régulièrement.

## 9. Règle de rentabilité : n'envoyer que si `p × Z × valeur > X`

Plafond : 500 offres par mois. Pour chaque prix d'offre `X` et efficacité `Z`, on compte les offres envoyées et le gain sur la validation.

In [9]:
p_val = probas['arbres']   # modèle retenu : mieux calibré et meilleur sur les 3 périodes

rows = []
for x in [1, 2, 3, 5]:
    for z in [0.2, 0.3, 0.5]:
        v = val.assign(p=p_val, gain_attendu=p_val * z * val.valeur_an_eur - x)
        cibles = v[v.gain_attendu > 0].nlargest(BUDGET, 'gain_attendu')
        gain_reel = (cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * x
        rows.append({'X (€)': x, 'Z': f'{z:.0%}', 'offres envoyées': len(cibles),
                     'churners ciblés': int(cibles.churn.sum()), 'gain réel (€)': round(gain_reel)})
carte = pd.DataFrame(rows)
carte.pivot(index='X (€)', columns='Z', values=['offres envoyées', 'gain réel (€)'])

offres envoyées           gain réel (€)           
Z                 20%  30%  50%           20%  30%   50%
X (€)                                                   
1                 500  500  500           416  873  1789
2                 313  490  500           143  393  1289
3                 174  313  500            61  214   789
5                  76  150  313            53  114   357

**Lecture :** avec la règle de rentabilité, le gain est **positif dans tous les scénarios**, mais modeste pour une offre chère : à 5 € et Z = 30 %, +114 €/mois sur 150 offres.

- La clé, c'est d'envoyer **moins d'offres, mieux ciblées** : à 5 € et Z = 30 %, 150 offres rapportent +114 €, alors que 500 offres aux plus risqués perdent environ 2 000 €.
- **Le prix de l'offre compte plus que le modèle** : à 2 € (Z = 30 %), +393 € ; à 1 €, +873 €.

**Recommandation provisoire :** on lance sous condition, avec une offre à 2 € maximum. Les arbres ciblent les clients à forte valeur et à risque, et on n'envoie que si c'est rentable client par client. Le premier mois, on teste avec un groupe témoin pour mesurer Z.

## 10. Réglage des arbres (sur la validation uniquement)

Trois réglages :
- **profondeur** : nombre de questions successives dans chaque arbre ;
- **taux d'apprentissage** : part de chaque correction qu'on accepte ;
- **nombre d'arbres**.

On essaie toutes les combinaisons et on garde celle qui a la meilleure PR-AUC sur la validation. Le gain en € (règle de rentabilité, X = 2 €, Z = 30 %) est affiché pour contrôle.

In [10]:
def gain_regle(p, data, x=2, z=0.30):
    v = data.assign(p=p, gain_attendu=p * z * data.valeur_an_eur - x)
    cibles = v[v.gain_attendu > 0].nlargest(BUDGET, 'gain_attendu')
    return (cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * x

essais = []
for profondeur in [3, 4, 6]:
    for taux in [0.03, 0.05, 0.1]:
        for nb_arbres in [100, 300]:
            m = HistGradientBoostingClassifier(max_depth=profondeur, learning_rate=taux,
                                               max_iter=nb_arbres, random_state=0)
            p = m.fit(fit[FEATURES], fit.churn).predict_proba(val[FEATURES])[:, 1]
            essais.append({'profondeur': profondeur, 'taux': taux, 'nb arbres': nb_arbres,
                           'PR-AUC': round(average_precision_score(val.churn, p), 4),
                           'gain € (X=2, Z=30%)': round(gain_regle(p, val))})
essais = pd.DataFrame(essais).sort_values('PR-AUC', ascending=False)
MEILLEUR = essais.iloc[0][['profondeur', 'taux', 'nb arbres']].to_dict()
print('Réglage retenu :', MEILLEUR)
essais.head(8)

Réglage retenu : {'profondeur': 3.0, 'taux': 0.03, 'nb arbres': 300.0}


,profondeur,taux,nb arbres,PR-AUC,"gain € (X=2, Z=30%)"
1,3,0.03,300,0.4725,452
4,3,0.10,100,0.4710,456
2,3,0.05,100,0.4702,384
0,3,0.03,100,0.4696,284
6,4,0.03,100,0.4682,319
10,4,0.10,100,0.4673,449
3,3,0.05,300,0.4669,457
7,4,0.03,300,0.4669,425


**Lecture :** le meilleur réglage est profondeur 3, taux 0,03, 300 arbres : PR-AUC 0,473 contre 0,465 avant, et gain de +452 € contre +393 €. Les meilleures combinaisons sont très proches : le gain du réglage est réel mais modeste. Des arbres peu profonds limitent l'apprentissage par cœur.

# 11. TEST — ouvert ici, une seule fois (25/11/2026)

**Choix figés avant l'ouverture :** churn à 60 jours ; clients actifs d'au moins 60 jours d'ancienneté ; 30 features ; arbres boostés (profondeur 3, taux 0,03, 300 arbres) ; entraînement final sur fit et validation réunis (coupures du 14/11/2018 et du 14/01/2019, labels tous connus avant le 15/03/2019) ; règle de rentabilité plafonnée à 500 offres.

Rien n'est modifié après cette section.

In [11]:
test = build_dataset(CUTOFFS['test'])
train = pd.concat([fit, val], ignore_index=True)
print(f'entraînement final : {len(train)} lignes ; test : {len(test)} clients, churn {test.churn.mean():.1%}')

final_trees = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.03, max_iter=300, random_state=0)
final_logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
p_trees = final_trees.fit(train[FEATURES], train.churn).predict_proba(test[FEATURES])[:, 1]
p_logit = final_logit.fit(train[FEATURES], train.churn).predict_proba(test[FEATURES])[:, 1]
test = test.assign(p=p_trees)

entraînement final : 18753 lignes ; test : 13068 clients, churn 13.0%


## 11.1 Qui repère le mieux les partants ? (500 clients les plus à risque)

In [12]:
methodes = {'hasard': None, 'baseline (récence)': test.recence_j, 'logistique': p_logit, 'arbres': p_trees}
rows = []
for nom, s in methodes.items():
    if s is None:
        rows.append({'méthode': nom, 'PR-AUC': round(test.churn.mean(), 3),
                     'churners dans le top 500': round(test.churn.mean() * BUDGET)})
        continue
    top = test.assign(s=s).nlargest(BUDGET, 's')
    rows.append({'méthode': nom, 'PR-AUC': round(average_precision_score(test.churn, s), 3),
                 'churners dans le top 500': int(top.churn.sum())})
pd.DataFrame(rows)

,méthode,PR-AUC,churners dans le top 500
0,hasard,0.130,65
1,baseline (récence),0.372,246
2,logistique,0.425,279
3,arbres,0.466,297


**Lecture :** sur le test, les arbres restent les meilleurs : 297 vrais partants dans le top 500, contre 279 pour la logistique, 246 pour la baseline et 65 au hasard. Le classement est le même qu'en validation : **le modèle est stable**.

## 11.2 Estimation en euros : 5 façons de choisir les clients

| Stratégie | Qui reçoit l'offre |
|---|---|
| A. Baseline récence | les 500 clients inactifs depuis le plus longtemps |
| B. Gros dépensiers (sans modèle) | les 500 clients à plus forte valeur |
| C. Risque (arbres) | les 500 clients au `p` le plus haut |
| D. Perte attendue (arbres) | les 500 clients au `p × valeur` le plus haut |
| E. Règle de rentabilité (arbres) | seulement si `p × Z × valeur > X`, 500 maximum |

Gain réel = Σ (churners ciblés × Z × leur valeur) − offres × X.

In [13]:
def gain_reel(cibles, x, z):
    return round((cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * x)

rows = []
for x in [1, 2, 5]:
    for z in [0.2, 0.3, 0.5]:
        strategies = {
            'A. baseline récence': test.nlargest(BUDGET, 'recence_j'),
            'B. gros dépensiers': test.nlargest(BUDGET, 'valeur_an_eur'),
            'C. risque (arbres)': test.nlargest(BUDGET, 'p'),
            'D. perte attendue (arbres)': test.assign(k=test.p * test.valeur_an_eur).nlargest(BUDGET, 'k'),
        }
        regle = test.assign(g=test.p * z * test.valeur_an_eur - x)
        strategies['E. règle de rentabilité'] = regle[regle.g > 0].nlargest(BUDGET, 'g')
        for nom, cibles in strategies.items():
            rows.append({'X (€)': x, 'Z': f'{z:.0%}', 'stratégie': nom, 'offres': len(cibles),
                         'churners ciblés': int(cibles.churn.sum()), 'gain €': gain_reel(cibles, x, z)})
estimations = pd.DataFrame(rows)
estimations.pivot_table(index='stratégie', columns=['X (€)', 'Z'], values='gain €')

X (€)                           1                      2                  \
Z                             20%     30%     50%    20%     30%     50%   
stratégie                                                                  
A. baseline récence        -245.0  -118.0   137.0 -745.0  -618.0  -363.0   
B. gros dépensiers          409.0   864.0  1773.0  -91.0   364.0  1273.0   
C. risque (arbres)         -315.0  -222.0   -36.0 -815.0  -722.0  -536.0   
D. perte attendue (arbres)  975.0  1712.0  3187.0  475.0  1212.0  2687.0   
E. règle de rentabilité     975.0  1712.0  3187.0  712.0  1338.0  2687.0   

X (€)                            5                  
Z                              20%     30%     50%  
stratégie                                           
A. baseline récence        -2245.0 -2118.0 -1863.0  
B. gros dépensiers         -1591.0 -1136.0  -227.0  
C. risque (arbres)         -2315.0 -2222.0 -2036.0  
D. perte attendue (arbres) -1025.0  -288.0  1187.0  
E. règle de rentabilité      554.0   925.0  1780.0

**Lecture — la question « prioriser les gros dépensiers » :**
- **Viser les plus risqués (C) perd de l'argent dans tous les scénarios**, et même plus que la baseline. Les clients les plus risqués sont des clients dormants qui ne rapportent presque rien.
- **Viser les gros dépensiers sans modèle (B) fait déjà mieux que viser les plus risqués** : la valeur compte plus que le risque.
- **Combiner les deux (D, `p × valeur`) double ou triple le gain** de B.
- **La règle de rentabilité (E) est la seule positive dans les 9 scénarios.** À 5 € l'offre, elle envoie seulement 70 à 265 offres et reste gagnante (+554 à +1 780 €), alors que toutes les autres stratégies perdent de l'argent.

In [14]:
# Détail de la règle de rentabilité (stratégie E)
estimations[estimations.stratégie == 'E. règle de rentabilité'].pivot_table(
    index='X (€)', columns='Z', values=['offres', 'churners ciblés', 'gain €'])

churners ciblés             gain €                 offres              
Z                 20%   30%   50%    20%     30%     50%    20%    30%    50%
X (€)                                                                        
1                89.0  89.0  89.0  975.0  1712.0  3187.0  500.0  500.0  500.0
2                55.0  74.0  89.0  712.0  1338.0  2687.0  265.0  390.0  500.0
5                28.0  38.0  55.0  554.0   925.0  1780.0   70.0  132.0  265.0

In [15]:
# Stabilité : validation contre test (arbres réglés, règle de rentabilité, X = 2 €, Z = 30 %)
arbres_regles = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.03, max_iter=300, random_state=0)
p_validation = arbres_regles.fit(fit[FEATURES], fit.churn).predict_proba(val[FEATURES])[:, 1]
print(f'validation : gain {gain_regle(p_validation, val):.0f} €, churn {val.churn.mean():.1%}')
print(f'test       : gain {gain_regle(test.p, test):.0f} €, churn {test.churn.mean():.1%}')

validation : gain 452 €, churn 14.1%
test       : gain 1338 €, churn 13.0%


**Lecture :** le gain sur le test (+1 338 €) est près de 3 fois celui de la validation (+452 €), pour le même scénario. Deux raisons : le modèle final est entraîné sur deux périodes au lieu d'une, et la période testée est peut-être plus favorable.

On ne présente pas +1 338 € comme le chiffre promis. **On annonce une fourchette : de +450 à +1 350 € par mois (X = 2 €, Z = 30 %).** L'écart entre les deux montre l'incertitude.

# 12. Où le modèle se trompe (test)

## 12.1 Par groupe : part des partants repérés dans les 500 clients les plus à risque

In [16]:
t = test.copy()
t['repéré'] = t.index.isin(t.nlargest(BUDGET, 'p').index)
colonnes_pays = [c for c in t.columns if c.startswith('pays_')]
t['pays'] = t[colonnes_pays].idxmax(axis=1).str.replace('pays_', '')
t['ancienneté'] = pd.cut(t.anciennete_j, [0, 180, 365, 10_000], labels=['2-6 mois', '6-12 mois', '> 1 an'])
t['activité (tx 90 j)'] = pd.qcut(t.nb_tx_90j, 4, labels=['faible', 'moyenne', 'forte', 'très forte'])
t['valeur'] = pd.qcut(t.valeur_an_eur.rank(method='first'), 4, labels=['petite', 'moyenne', 'grande', 'très grande'])

def par_groupe(col):
    g = t.groupby(col, observed=True).apply(lambda d: pd.Series({
        'clients': len(d), 'taux de churn': f'{d.churn.mean():.0%}', 'partants': int(d.churn.sum()),
        'partants repérés': f'{d[d.churn == 1].repéré.mean():.0%}',
        'PR-AUC': round(average_precision_score(d.churn, d.p), 2) if d.churn.nunique() > 1 else None}),
        include_groups=False)
    return g

for col in ['pays', 'ancienneté', 'activité (tx 90 j)', 'valeur']:
    print(f'--- {col}'); print(par_groupe(col).to_string()); print()

--- pays
       clients taux de churn  partants partants repérés  PR-AUC
pays                                                           
FR        1387           15%       206              17%    0.49
GB        3842           16%       603              19%    0.49
IE        1013            7%        68               7%    0.38
PL        1649           12%       201              14%    0.39
RO         886            5%        45              11%    0.30
autre     4291           14%       581              19%    0.49

--- ancienneté
            clients taux de churn  partants partants repérés  PR-AUC
ancienneté                                                          
2-6 mois       4998           14%       710              19%    0.49
6-12 mois      6083           12%       740              16%    0.43
> 1 an         1987           13%       254              17%    0.51

--- activité (tx 90 j)
                    clients taux de churn  partants partants repérés  PR-AUC
activité (tx 90 j

**Lecture :**
- **Le modèle repère surtout les partants « qui s'éteignent »** : activité faible (26 % repérés), valeur faible (28 %).
- **Il rate presque tous les départs de clients actifs** : seulement 4 à 5 % des partants à activité moyenne, forte ou très forte sont repérés, et 3 à 4 % pour les valeurs moyenne et grande.
- **Pays :** plus faible en Irlande (7 % repérés) et en Roumanie (11 %), où le churn est rare.
- **Ancienneté :** pas de faiblesse marquée.

→ **Limite principale : le modèle ne voit pas venir les départs brutaux de clients encore actifs.** Ce sont pourtant ceux qui coûtent le plus.

## 12.2 Les erreurs les plus coûteuses

**Partants ratés** : clients de grande valeur partis alors que le modèle les jugeait peu à risque.

In [17]:
cols = ['p', 'valeur_an_eur', 'recence_j', 'nb_tx_30j', 'nb_tx_90j', 'tendance', 'nb_categories_90j', 'nb_contacts', 'anciennete_j']
rates = t[(t.churn == 1)].assign(perte=lambda d: d.valeur_an_eur * (1 - d.p)).nlargest(10, 'perte')
rates[cols].round(2)

,p,valeur_an_eur,recence_j,nb_tx_30j,nb_tx_90j,tendance,nb_categories_90j,nb_contacts,anciennete_j
5196,0.02,409.21,0,10.0,18,1.43,3.0,9,232
1289,0.20,431.32,22,3.0,68,0.13,17.0,0,338
2482,0.22,427.48,6,18.0,18,2.57,3.0,7,320
7826,0.27,409.28,48,0.0,21,0.00,1.0,76,92
67,0.09,250.19,9,5.0,15,0.83,1.0,0,395
5533,0.13,258.80,14,45.0,46,2.76,13.0,7,391
11985,0.18,273.53,24,1.0,58,0.05,16.0,29,87
5027,0.14,254.33,26,3.0,19,0.41,6.0,24,428
6359,0.15,256.32,4,1.0,2,0.60,0.0,0,411
10534,0.32,256.32,27,3.0,7,0.90,0.0,0,148


**Lecture :** les partants ratés les plus coûteux (250 à 430 €/an) étaient **encore actifs** : récence de 0 à 26 jours, souvent une activité en hausse. Rien dans leurs transactions n'annonçait le départ (déménagement, changement de banque, motif hors des données ?). Un modèle basé sur les transactions ne peut pas les voir.

**Fausses alertes** : clients jugés les plus à risque, qui sont pourtant restés.

In [18]:
t[t.churn == 0].nlargest(10, 'p')[cols].round(2)

,p,valeur_an_eur,recence_j,nb_tx_30j,nb_tx_90j,tendance,nb_categories_90j,nb_contacts,anciennete_j
13033,0.78,0.0,34,0.0,1,0.00,0.0,0,90
13032,0.74,0.0,37,0.0,1,0.00,0.0,0,213
1265,0.72,0.0,56,0.0,2,0.00,0.0,0,322
3362,0.69,0.0,51,0.0,1,0.00,0.0,0,294
10187,0.69,0.0,53,0.0,1,0.00,0.0,0,148
7689,0.69,0.0,50,0.0,6,0.00,0.0,3,76
10662,0.69,0.0,58,0.0,20,0.00,0.0,0,75
6426,0.68,0.0,48,0.0,1,0.00,0.0,0,397
6224,0.68,0.0,20,1.0,1,0.75,0.0,0,395
10141,0.68,0.0,54,0.0,1,0.00,0.0,0,211


**Lecture :** les fausses alertes sont des clients dormants (1 transaction en 90 jours, valeur 0 €) qui ne sont pas partis selon notre définition. **Elles ne coûtent rien avec la règle de rentabilité** : leur valeur est nulle, donc elles ne reçoivent jamais d'offre. C'est l'avantage de combiner risque et valeur.

# 13. Vue d'ensemble des données (pour comprendre, pas pour le rendu)

Données d'**entraînement** uniquement (coupures du 14/11/2018 et du 14/01/2019). Le test n'est pas utilisé ici.

⚠️ Ces tableaux montrent ce qui **distingue** les partants, pas ce qui **cause** leur départ.

## 13.1 Dictionnaire des données : chaque donnée utilisée

In [19]:
dico = pd.DataFrame([
    # (colonne, famille, ce que c'est, fichier source, fenêtre)
    ('anciennete_j', 'Profil', 'jours depuis l\'inscription', 'users', 'à la coupure'),
    ('recence_j', 'Activité', 'jours depuis la dernière transaction', 'transactions', 'tout le passé'),
    ('nb_tx_30j', 'Activité', 'nombre de transactions', 'transactions', '30 j'),
    ('nb_tx_90j', 'Activité', 'nombre de transactions', 'transactions', '90 j'),
    ('tendance', 'Activité', 'rythme sur 30 j / rythme moyen sur 90 j (>1 accélère, <1 ralentit)', 'transactions', '30 et 90 j'),
    ('montant_90j', 'Montants', 'somme dépensée et reçue, transactions terminées, plafonnée à 851 $', 'transactions', '90 j'),
    ('panier_moyen_90j', 'Montants', 'montant moyen d\'une transaction terminée', 'transactions', '90 j'),
    ('part_card_payment', 'Types', 'part des transactions qui sont des paiements carte', 'transactions', '90 j'),
    ('part_transfer', 'Types', 'part des virements', 'transactions', '90 j'),
    ('part_topup', 'Types', 'part des recharges du compte', 'transactions', '90 j'),
    ('part_exchange', 'Types', 'part des opérations de change', 'transactions', '90 j'),
    ('part_atm', 'Types', 'part des retraits au distributeur', 'transactions', '90 j'),
    ('taux_echec_90j', 'Échecs', 'part des transactions refusées, échouées, annulées ou reversées', 'transactions', '90 j'),
    ('nb_pays_marchands_90j', 'Marchands', 'nombre de pays différents où la carte a payé', 'transactions (carte)', '90 j'),
    ('nb_categories_90j', 'Marchands', 'nombre de catégories de commerces différentes', 'transactions (carte)', '90 j'),
    ('nb_notifs_90j', 'Notifications', 'nombre de notifications reçues', 'notifications', '90 j'),
    ('taux_echec_notifs_90j', 'Notifications', 'part des notifications non délivrées', 'notifications', '90 j'),
    ('age', 'Profil', 'âge à la coupure', 'users', 'fixe'),
    ('crypto', 'Profil', 'crypto activée (photo à l\'export)', 'users', 'export mai 2019'),
    ('nb_contacts', 'Profil', 'nombre de contacts dans l\'app (photo à l\'export)', 'users', 'export mai 2019'),
    ('optin_push / optin_email', 'Profil', 'accepte le marketing (0 si inconnu)', 'users', 'export mai 2019'),
    ('optin_*_inconnu', 'Profil', '1 si l\'opt-in est inconnu', 'users', 'export mai 2019'),
    ('pays_GB, _FR, _PL, _IE, _RO, _autre', 'Profil', 'pays du client (5 principaux + autre)', 'users', 'fixe'),
    ('valeur_an_eur', 'Décision', 'frais + 0,3 % des paiements carte, × 4, en € : ce que le client rapporte par an', 'transactions', '90 j'),
    ('churn', 'Label', '1 si aucune transaction dans les 60 j après la coupure', 'transactions', '60 j APRÈS'),
], columns=['colonne', 'famille', 'ce que c\'est', 'source', 'fenêtre'])
dico['connue à la décision ?'] = np.where(dico.famille == 'Label', 'non (c\'est la réponse)',
                                np.where(dico.fenêtre == 'export mai 2019', 'en partie (photo à l\'export)', 'oui'))
pd.set_option('display.max_colwidth', 90)
dico

,colonne,famille,ce que c'est,source,fenêtre,connue à la décision ?
0,anciennete_j,Profil,jours depuis l'inscription,users,à la coupure,oui
1,recence_j,Activité,jours depuis la dernière transaction,transactions,tout le passé,oui
2,nb_tx_30j,Activité,nombre de transactions,transactions,30 j,oui
3,nb_tx_90j,Activité,nombre de transactions,transactions,90 j,oui
4,tendance,Activité,"rythme sur 30 j / rythme moyen sur 90 j (>1 accélère, <1 ralentit)",transactions,30 et 90 j,oui
5,montant_90j,Montants,"somme dépensée et reçue, transactions terminées, plafonnée à 851 $",transactions,90 j,oui
6,panier_moyen_90j,Montants,montant moyen d'une transaction terminée,transactions,90 j,oui
7,part_card_payment,Types,part des transactions qui sont des paiements carte,transactions,90 j,oui
8,part_transfer,Types,part des virements,transactions,90 j,oui
9,part_topup,Types,part des recharges du compte,transactions,90 j,oui


## 13.2 Trois groupes de dépense : qu'est-ce qui distingue les partants ?

Les clients sont coupés en 3 groupes de même taille selon le montant passé sur 90 jours : **petits**, **moyens** et **gros** dépensiers. Dans chaque groupe, on compare ceux qui **restent** et ceux qui **partent** (médianes, ou moyennes pour les parts).

In [20]:
train = pd.concat([fit, val], ignore_index=True)
train['groupe'] = pd.qcut(train.montant_90j.rank(method='first'), 3, labels=['petits', 'moyens', 'gros'])
train['statut'] = np.where(train.churn == 1, 'partent', 'restent')

resume = train.groupby('groupe', observed=True).agg(
    clients=('churn', 'size'), taux_churn=('churn', 'mean'),
    depense_mediane_90j=('montant_90j', 'median'), valeur_moyenne_an=('valeur_an_eur', 'mean'))
resume['part du revenu total'] = (train.groupby('groupe', observed=True).valeur_an_eur.sum() / train.valeur_an_eur.sum())
resume['revenu perdu avec les partants'] = train[train.churn == 1].groupby('groupe', observed=True).valeur_an_eur.sum() / train[train.churn == 1].valeur_an_eur.sum()
resume.round(2)

,clients,taux_churn,depense_mediane_90j,valeur_moyenne_an,part du revenu total,revenu perdu avec les partants
groupe,,,,,,
petits,6251,0.27,88.08,7.16,0.12,0.37
moyens,6251,0.13,947.11,11.89,0.21,0.28
gros,6251,0.06,4137.94,38.48,0.67,0.35


In [21]:
indicateurs = {
    'recence_j': ('jours depuis la dernière transaction', 'median'),
    'nb_tx_30j': ('transactions sur 30 j', 'median'),
    'tendance': ('tendance (<1 = ralentit)', 'median'),
    'panier_moyen_90j': ('panier moyen ($)', 'median'),
    'taux_echec_90j': ('taux d\'échec des paiements', 'mean'),
    'nb_categories_90j': ('catégories de commerces', 'median'),
    'nb_pays_marchands_90j': ('pays marchands', 'median'),
    'part_card_payment': ('part de paiements carte', 'mean'),
    'part_topup': ('part de recharges', 'mean'),
    'part_transfer': ('part de virements', 'mean'),
    'nb_contacts': ('contacts dans l\'app', 'median'),
    'nb_notifs_90j': ('notifications reçues', 'median'),
    'anciennete_j': ('ancienneté (jours)', 'median'),
    'age': ('âge', 'median'),
}
lignes = []
for col, (libelle, f) in indicateurs.items():
    t = train.groupby(['groupe', 'statut'], observed=True)[col].agg(f)
    t.name = libelle
    lignes.append(t)
profil = pd.DataFrame(lignes).round(2)
profil.columns = [f'{g} — {s}' for g, s in profil.columns]
profil

,petits — partent,petits — restent,moyens — partent,moyens — restent,gros — partent,gros — restent
jours depuis la dernière transaction,30.00,10.00,26.00,4.00,21.00,1.00
transactions sur 30 j,0.00,2.00,1.00,11.00,3.00,31.00
tendance (<1 = ralentit),0.00,0.67,0.16,0.86,0.30,0.94
panier moyen ($),10.35,11.81,50.19,30.93,85.62,55.25
taux d'échec des paiements,0.30,0.25,0.15,0.15,0.13,0.11
catégories de commerces,0.00,2.00,5.00,7.00,10.00,15.00
pays marchands,0.00,1.00,2.00,2.00,2.00,3.00
part de paiements carte,0.41,0.48,0.49,0.54,0.52,0.54
part de recharges,0.28,0.25,0.22,0.20,0.20,0.16
part de virements,0.16,0.14,0.11,0.13,0.10,0.15


## 13.3 Par pays et par groupe de dépense

In [22]:
pays_cols = [c for c in train.columns if c.startswith('pays_')]
train['pays'] = train[pays_cols].idxmax(axis=1).str.replace('pays_', '')

par_pays = train.groupby('pays').agg(clients=('churn', 'size'), taux_churn=('churn', 'mean'),
                                     valeur_moyenne_an=('valeur_an_eur', 'mean'),
                                     echec_paiements=('taux_echec_90j', 'mean'),
                                     contacts_median=('nb_contacts', 'median'))
par_pays['revenu perdu (€/an)'] = train[train.churn == 1].groupby('pays').valeur_an_eur.sum()
croise = train.pivot_table(index='pays', columns='groupe', values='churn', aggfunc='mean', observed=True)
croise.columns = [f'churn {g}' for g in croise.columns]
par_pays.join(croise).sort_values('revenu perdu (€/an)', ascending=False).round(2)

,clients,taux_churn,valeur_moyenne_an,echec_paiements,contacts_median,revenu perdu (€/an),churn petits,churn moyens,churn gros
pays,,,,,,,,,
GB,6239,0.19,21.51,0.14,5.0,9136.49,0.33,0.19,0.08
autre,5825,0.14,20.06,0.19,9.0,5531.81,0.26,0.10,0.04
FR,2122,0.16,20.51,0.21,4.0,2461.23,0.28,0.12,0.06
PL,2373,0.14,8.43,0.21,6.0,905.97,0.22,0.10,0.04
IE,1209,0.07,19.22,0.13,23.0,610.14,0.19,0.05,0.02
RO,985,0.07,22.19,0.17,39.0,175.20,0.16,0.03,0.02


### Lecture

**Les gros dépensiers sont l'enjeu.** Un tiers des clients, mais 67 % du revenu. Leur taux de churn est faible (6 %), et pourtant ils pèsent 35 % du revenu perdu avec les partants.

**Ce qui distingue les partants, groupe par groupe :**
- **Dans tous les groupes**, les partants avaient déjà décroché avant la coupure : 21 à 30 jours sans transaction (contre 1 à 10 pour ceux qui restent), presque plus de transactions sur 30 jours, une tendance qui s'effondre.
- **Gros dépensiers qui partent** : moins de transactions, mais un **panier moyen plus élevé** (86 $ contre 55 $) et moins de virements. Ils utilisaient Revolut pour quelques gros achats, pas comme compte du quotidien. Ils ont aussi **beaucoup moins de contacts** (2 contre 12).
- **Les échecs de paiement ne sont pas la raison principale** : un écart marqué seulement chez les petits dépensiers (30 % contre 25 %), et presque aucun ailleurs.
- **Les contacts** reviennent partout : les clients qui ont un réseau dans l'app restent.

**Par pays :**
- **Royaume-Uni** : le churn le plus élevé dans chaque groupe (8 % chez les gros dépensiers, contre 2 % en Irlande et en Roumanie), et la moitié du revenu perdu.
- **Irlande et Roumanie** : peu de churn et beaucoup de contacts (23 et 39 en médiane). Cela va dans le sens de « le réseau retient ».
- **Pologne** : valeur par client 2 fois plus faible (8 € par an).

**Idée d'action** (hors du périmètre du modèle, à présenter comme une piste) : chez les gros dépensiers, un client qui ralentit **et** qui a peu de contacts est le profil à surveiller en priorité, surtout au Royaume-Uni.